# TalentTrack — Atoti DataMart
**Kelompok:** Daffa Ahmad Pangreksa · Halilatunnisa · Naura Kanaya Putri M.  
**Kelas:** 2024-INT | Data Warehouse | UNESA  

Notebook ini membangun DataMart menggunakan **Atoti** dengan 3 cube OLAP:
1. `JobPostings` — analisis umum posting pekerjaan
2. `SkillDemand` — analisis permintaan skill per kategori/region/waktu
3. `SkillForecast` — hasil forecasting permintaan skill 8 minggu ke depan

---
**Prasyarat:**
```bash
pip install atoti[all] sqlalchemy psycopg2-binary python-dotenv pandas
```
Pastikan file `.env` ada di folder `talenttrack/` berisi credentials Supabase.

In [ ]:
import sys
import os
from pathlib import Path

ROOT = Path("__file__").resolve().parent.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from dotenv import load_dotenv
load_dotenv(ROOT / ".env")
print("Environment loaded")

In [ ]:
import pandas as pd
import numpy as np
import atoti as tt
from sqlalchemy import create_engine

DB_URL = (
    f"postgresql://{os.getenv('PGUSER')}:{os.getenv('PGPASSWORD')}"
    f"@{os.getenv('PGHOST')}:{os.getenv('PGPORT', 5432)}/{os.getenv('PGDATABASE')}"
)
engine = create_engine(DB_URL, pool_pre_ping=True)
print("DB engine created")

## 1. Load Data dari PostgreSQL

In [ ]:
fact_df = pd.read_sql("""
    SELECT
        f.job_id,
        f.posting_count,
        f.job_age_days,
        CASE WHEN f.has_salary THEN 1 ELSE 0 END AS has_salary,
        CASE WHEN f.is_remote  THEN 1 ELSE 0 END AS is_remote,
        f.salary_min,
        f.salary_max,
        dt.date,
        dt.week,
        dt.month,
        dt.quarter,
        dt.year,
        dt.week_label,
        dl.city,
        dl.province_state,
        dl.country,
        dl.global_region,
        dc.company_name,
        dc.industry_sector,
        dc.company_size_category,
        dp.normalized_job_title,
        dp.job_level,
        dp.job_category,
        dpl.platform_name
    FROM fact_job_posting f
    JOIN dim_time dt      ON f.time_id     = dt.time_id
    JOIN dim_location dl  ON f.location_id = dl.location_id
    JOIN dim_company dc   ON f.company_id  = dc.company_id
    JOIN dim_position dp  ON f.position_id = dp.position_id
    JOIN dim_platform dpl ON f.platform_id = dpl.platform_id;
""", engine)

fact_df["date"] = pd.to_datetime(fact_df["date"])
print(f"fact_job_posting: {len(fact_df):,} rows")
fact_df.head(3)

In [ ]:
skill_df = pd.read_sql("""
    SELECT
        b.job_id,
        b.extraction_confidence,
        ds.skill_name,
        ds.skill_type,
        ds.skill_domain
    FROM bridge_job_skill b
    JOIN dim_skill ds ON b.skill_id = ds.skill_id;
""", engine)

print(f"bridge + dim_skill: {len(skill_df):,} rows")
skill_df.head(3)

In [ ]:
forecast_df = pd.read_sql("""
    SELECT
        fsd.forecast_week_label,
        fsd.forecast_year,
        fsd.forecast_week,
        ds.skill_name,
        ds.skill_type,
        ds.skill_domain,
        fsd.job_category,
        fsd.global_region,
        fsd.predicted_count,
        fsd.lower_bound,
        fsd.upper_bound,
        fsd.model_name
    FROM forecast_skill_demand fsd
    JOIN dim_skill ds ON fsd.skill_id = ds.skill_id;
""", engine)

print(f"forecast_skill_demand: {len(forecast_df):,} rows")
forecast_df.head(3)

In [ ]:
skill_fact_df = skill_df.merge(fact_df, on="job_id", how="left")
skill_fact_df["salary_min"]  = pd.to_numeric(skill_fact_df["salary_min"],  errors="coerce")
skill_fact_df["salary_max"]  = pd.to_numeric(skill_fact_df["salary_max"],  errors="coerce")
skill_fact_df["salary_midpoint"] = (
    (skill_fact_df["salary_min"].fillna(0) + skill_fact_df["salary_max"].fillna(0)) / 2
).replace(0, np.nan)

print(f"skill_fact merged: {len(skill_fact_df):,} rows")
skill_fact_df.head(3)

## 2. Buat Sesi Atoti

In [ ]:
session = tt.create_session(
    name="TalentTrack DataMart",
    config={
        "port": 9090,
        "authentication": {"mechanism": "DISABLED"},
    },
)
print(f"Atoti session created → http://localhost:9090/ui")

## 3. Cube 1: JobPostings
Analisis umum: volume posting, remote rate, salary coverage, per platform/region/waktu.

In [ ]:
job_store = session.read_pandas(
    fact_df,
    store_name="job_postings",
    keys=["job_id"],
)

job_cube = session.create_cube(job_store, name="JobPostings")
h = job_cube.hierarchies
m = job_cube.measures

h["Time"]     = [job_store["year"], job_store["quarter"],
                  job_store["month"], job_store["week_label"]]
h["Location"] = [job_store["global_region"], job_store["country"],
                  job_store["city"]]
h["Job"]      = [job_store["job_category"], job_store["job_level"],
                  job_store["normalized_job_title"]]
h["Company"]  = [job_store["industry_sector"], job_store["company_size_category"],
                  job_store["company_name"]]
h["Platform"] = [job_store["platform_name"]]

m["Total Postings"]      = tt.agg.sum(job_store["posting_count"])
m["Remote Postings"]     = tt.agg.sum(job_store["is_remote"])
m["Remote Rate"]         = m["Remote Postings"] / m["Total Postings"]
m["Avg Salary Min"]      = tt.agg.mean(job_store["salary_min"])
m["Avg Salary Max"]      = tt.agg.mean(job_store["salary_max"])
m["Avg Job Age Days"]    = tt.agg.mean(job_store["job_age_days"])
m["With Salary Count"]   = tt.agg.sum(job_store["has_salary"])
m["Salary Coverage Rate"] = m["With Salary Count"] / m["Total Postings"]

print("Cube 1 (JobPostings) created ✓")

## 4. Cube 2: SkillDemand
Analisis permintaan skill: skill apa paling banyak diminta, per kategori, region, waktu.

In [ ]:
skill_store = session.read_pandas(
    skill_fact_df,
    store_name="skill_facts",
    keys=["job_id", "skill_name"],
)

skill_cube = session.create_cube(skill_store, name="SkillDemand")
sh = skill_cube.hierarchies
sm = skill_cube.measures

sh["Skill"]    = [skill_store["skill_domain"], skill_store["skill_type"],
                   skill_store["skill_name"]]
sh["Job"]      = [skill_store["job_category"], skill_store["job_level"],
                   skill_store["normalized_job_title"]]
sh["Location"] = [skill_store["global_region"], skill_store["country"]]
sh["Time"]     = [skill_store["year"], skill_store["quarter"],
                   skill_store["month"], skill_store["week_label"]]
sh["Platform"] = [skill_store["platform_name"]]

sm["Skill Postings"]    = tt.agg.count(skill_store["job_id"])
sm["Avg Confidence"]    = tt.agg.mean(skill_store["extraction_confidence"])
sm["Avg Salary Midpoint"] = tt.agg.mean(skill_store["salary_midpoint"])

print("Cube 2 (SkillDemand) created ✓")

## 5. Cube 3: SkillForecast
Hasil forecasting Holt-Winters/Linear Trend: prediksi permintaan skill 8 minggu ke depan.

In [ ]:
forecast_store = session.read_pandas(
    forecast_df,
    store_name="skill_forecasts",
    keys=["forecast_week_label", "skill_name", "job_category", "global_region"],
)

forecast_cube = session.create_cube(forecast_store, name="SkillForecast")
fh = forecast_cube.hierarchies
fm = forecast_cube.measures

fh["Skill"]         = [forecast_store["skill_domain"], forecast_store["skill_type"],
                         forecast_store["skill_name"]]
fh["Job"]           = [forecast_store["job_category"]]
fh["Location"]      = [forecast_store["global_region"]]
fh["Forecast Time"] = [forecast_store["forecast_year"], forecast_store["forecast_week"],
                         forecast_store["forecast_week_label"]]
fh["Model"]         = [forecast_store["model_name"]]

fm["Predicted Postings"] = tt.agg.sum(forecast_store["predicted_count"])
fm["Forecast Lower"]     = tt.agg.sum(forecast_store["lower_bound"])
fm["Forecast Upper"]     = tt.agg.sum(forecast_store["upper_bound"])
fm["Forecast Range"]     = fm["Forecast Upper"] - fm["Forecast Lower"]

print("Cube 3 (SkillForecast) created ✓")

## 6. Widgets — Insight 1: Top 20 Skill Paling Banyak Diminta

In [ ]:
top_skills = skill_cube.query(
    skill_cube.measures["Skill Postings"],
    levels=[skill_cube.hierarchies["Skill"]["skill_name"]],
    include_totals=False,
).sort_values("Skill Postings", ascending=False).head(20)

print("Top 20 Skills by Posting Count:")
top_skills

## 7. Widgets — Insight 2: Tren Skill per Minggu

In [ ]:
skill_trend = skill_cube.query(
    skill_cube.measures["Skill Postings"],
    levels=[
        skill_cube.hierarchies["Time"]["week_label"],
        skill_cube.hierarchies["Skill"]["skill_name"],
    ],
    include_totals=False,
)
print(f"Skill trend data shape: {skill_trend.shape}")
skill_trend.head(10)

## 8. Widgets — Insight 3: Perbandingan Platform

In [ ]:
platform_comparison = job_cube.query(
    job_cube.measures["Total Postings"],
    job_cube.measures["Remote Rate"],
    job_cube.measures["Salary Coverage Rate"],
    levels=[
        job_cube.hierarchies["Platform"]["platform_name"],
        job_cube.hierarchies["Job"]["job_category"],
    ],
    include_totals=False,
)
print(f"Platform comparison shape: {platform_comparison.shape}")
platform_comparison.head(10)

## 9. Widgets — Insight 4: Forecasting Skill 8 Minggu ke Depan

In [ ]:
forecast_result = forecast_cube.query(
    forecast_cube.measures["Predicted Postings"],
    forecast_cube.measures["Forecast Lower"],
    forecast_cube.measures["Forecast Upper"],
    levels=[
        forecast_cube.hierarchies["Forecast Time"]["forecast_week_label"],
        forecast_cube.hierarchies["Skill"]["skill_name"],
    ],
    include_totals=False,
).sort_values("Predicted Postings", ascending=False)

print(f"Forecast result shape: {forecast_result.shape}")
forecast_result.head(15)

## 10. Widgets — Insight 5: Salary per Skill dan Job Level

In [ ]:
salary_by_skill = skill_cube.query(
    skill_cube.measures["Avg Salary Midpoint"],
    skill_cube.measures["Skill Postings"],
    levels=[
        skill_cube.hierarchies["Skill"]["skill_name"],
        skill_cube.hierarchies["Job"]["job_level"],
    ],
    include_totals=False,
).sort_values("Avg Salary Midpoint", ascending=False)

print(f"Salary by skill/level shape: {salary_by_skill.shape}")
salary_by_skill.head(15)

## 11. Widgets — Insight 6: Distribusi Geografis Posting

In [ ]:
geo_distribution = job_cube.query(
    job_cube.measures["Total Postings"],
    job_cube.measures["Remote Rate"],
    levels=[
        job_cube.hierarchies["Location"]["global_region"],
        job_cube.hierarchies["Location"]["country"],
        job_cube.hierarchies["Job"]["job_category"],
    ],
    include_totals=True,
)
print(f"Geo distribution shape: {geo_distribution.shape}")
geo_distribution.head(15)

## 12. OLAP CUBE Query (PostgreSQL)
Demonstrasi query CUBE langsung ke PostgreSQL untuk laporan.

In [ ]:
cube_query = pd.read_sql("""
    SELECT
        dt.year,
        dt.quarter,
        ds.skill_name,
        ds.skill_domain,
        dp.job_category,
        dl.global_region,
        COUNT(DISTINCT f.job_id) AS posting_count,
        AVG(f.salary_max)        AS avg_salary_max
    FROM fact_job_posting f
    JOIN bridge_job_skill b  ON f.job_id     = b.job_id
    JOIN dim_skill ds         ON b.skill_id   = ds.skill_id
    JOIN dim_time dt          ON f.time_id    = dt.time_id
    JOIN dim_position dp      ON f.position_id = dp.position_id
    JOIN dim_location dl      ON f.location_id = dl.location_id
    GROUP BY CUBE(
        dt.year, dt.quarter,
        ds.skill_name, ds.skill_domain,
        dp.job_category,
        dl.global_region
    )
    HAVING COUNT(DISTINCT f.job_id) > 0
    ORDER BY dt.year NULLS LAST, posting_count DESC
    LIMIT 1000;
""", engine)

print(f"CUBE query result: {len(cube_query):,} rows (including subtotals/grand totals)")
cube_query.head(10)

## 13. Performance Benchmark
Perbandingan query dengan dan tanpa Materialized View.

In [ ]:
import time

query_without_mv = """
    SELECT ds.skill_name, dt.week_label, COUNT(DISTINCT f.job_id) AS cnt
    FROM fact_job_posting f
    JOIN bridge_job_skill b ON f.job_id    = b.job_id
    JOIN dim_skill ds        ON b.skill_id  = ds.skill_id
    JOIN dim_time dt         ON f.time_id   = dt.time_id
    JOIN dim_position dp     ON f.position_id = dp.position_id
    JOIN dim_location dl     ON f.location_id = dl.location_id
    GROUP BY ds.skill_name, dt.week_label
    ORDER BY cnt DESC
    LIMIT 20;
"""

query_with_mv = """
    SELECT skill_name, week_label, SUM(posting_count) AS cnt
    FROM mv_weekly_skill_demand
    GROUP BY skill_name, week_label
    ORDER BY cnt DESC
    LIMIT 20;
"""

runs = 3
times_raw, times_mv = [], []

for _ in range(runs):
    t0 = time.time()
    pd.read_sql(query_without_mv, engine)
    times_raw.append(time.time() - t0)

    t0 = time.time()
    pd.read_sql(query_with_mv, engine)
    times_mv.append(time.time() - t0)

avg_raw = sum(times_raw) / runs
avg_mv  = sum(times_mv)  / runs
speedup = avg_raw / avg_mv if avg_mv > 0 else float("inf")

print(f"Without Materialized View : {avg_raw:.4f}s (avg of {runs} runs)")
print(f"With    Materialized View : {avg_mv:.4f}s  (avg of {runs} runs)")
print(f"Speedup                   : {speedup:.2f}x")

## 14. Buka Atoti UI
Jalankan cell di bawah, lalu buka browser di `http://localhost:9090/ui`

In [ ]:
print("Atoti DataMart running at: http://localhost:9090/ui")
print("Available cubes: JobPostings | SkillDemand | SkillForecast")
print("Press Stop/Interrupt kernel to shut down.")
session.wait()